In [38]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split


In [19]:
df = pd.read_csv('../data/combined_output.csv')

df.head()

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,...,10_day_max,10_day_min,30_day_max,30_day_min,day_of_week,numerical_sentiment,mean_sentiment_probability,percent_positive,percent_negative,percent_neutral
0,2026-01-16,620.25,629.0800,620.080,624.1762,17012516,619.731458,628.554076,619.561600,623.654376,...,629.0800,614.23,629.0800,614.23,5,-0.050847,0.837955,0.152542,0.203390,0.644068
1,2026-01-20,604.12,611.4000,600.000,607.8800,15040799,603.614943,610.888857,599.498388,607.371800,...,629.0800,600.00,629.0800,600.00,2,-0.122449,0.797898,0.142857,0.265306,0.591837
2,2026-01-21,612.96,618.2700,600.080,606.7400,14494655,612.447553,617.753114,599.578321,606.232753,...,629.0800,600.00,629.0800,600.00,3,0.127273,0.806921,0.236364,0.109091,0.654545
3,2026-01-22,647.63,660.5700,626.550,629.3500,21394669,647.088568,660.017750,626.026191,628.823850,...,660.5700,600.00,660.5700,600.00,4,0.038462,0.842375,0.230769,0.192308,0.576923
4,2026-01-23,658.76,666.4899,644.445,644.7700,22797723,658.209263,665.932701,643.906231,644.230959,...,666.4899,600.00,666.4899,600.00,5,-0.049180,0.773456,0.163934,0.213115,0.622951


In [54]:
class StockDataset(Dataset):
    def __init__(self, df, column_to_predict):
        super().__init__()
        self.data = df.to_numpy()
        self.idx_of_col_to_predict = df.columns.get_loc(column_to_predict)
        self.features = np.delete(self.data, [self.idx_of_col_to_predict, 0], axis=1)
        self.labels = self.data[:, self.idx_of_col_to_predict]

    def __len__(self):
        return self.data.shape[0]

    def create_sequence(self, seq_length):
        x_vals, y_vals = [], []
        for i in range(len(self.data) - seq_length):
            x = self.features[i:(i+seq_length)]
            y = self.labels[i + seq_length]
            x_vals.append(x)
            y_vals.append(y)
        return x_vals, y_vals

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [55]:
train_df, test_df = train_test_split(df, test_size=0.2, shuffle=False)

In [56]:
train_dataset = StockDataset(train_df, 'close')

train_dataset.__getitem__(0)

(array([629.08, 620.08, 624.1762, 17012516, 619.7314582587, 628.5540761973,
        619.5616003822, 623.6543758748, 17012516, 0.0, 1.0, 620.8, 624.17,
        614.23, 618.48, 13076058.0, 629.08, 614.23, 629.08, 614.23, 629.08,
        614.23, 5, -0.0508474576271186, 0.8379549252784858,
        0.1525423728813559, 0.2033898305084746, 0.6440677966101694],
       dtype=object),
 620.25)

In [61]:
seq_length = 8

X_train, y_train = train_dataset.create_sequence(seq_length)

dataset_train = TensorDataset(
    torch.from_numpy(train_dataset.features.astype(np.float32)),
    torch.from_numpy(train_dataset.labels.astype(np.float32))
)

In [62]:

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=train_dataset.features.shape[1],
            hidden_size=256,
            num_layers=2,
            batch_first=True
        )
        self.fc = nn.Linear(256, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 256)
        c0 = torch.zeros(2, x.size(0), 256)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

In [72]:
model = Model()
epochs = 3

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train()

for epoch in range(epochs):
    for seqs, labels in dataset_train:
        seqs = seqs.view(1, 58, 28)
        outputs = model(seqs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


RuntimeError: shape '[1, 58, 28]' is invalid for input of size 28